In [ ]:
import sys, os

sys.path.append(os.path.dirname(os.getcwd()))

import h5py
import tqdm
import matplotlib.pyplot as plt

plt.rcParams.update({"mathtext.fontset": "cm"})

import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

from flax import nnx
import optax

import models.time_solver as imex
from models.sph_mhd import (
    SphericalMHD,
    dynamical_solver,
    integral,
    kinetic_energy,
)

In [ ]:
eq = SphericalMHD(
    Ra=100, E=1e-3, Pr=1, radius_ratio=0.35, l_max=31, m_max=7, m_res=4, n_r=84
)
eq

In [ ]:
with h5py.File("../data/calibration/calibration_samples.h5", "r") as f:
    dt = f.attrs["dt"]
    steps = f.attrs["steps"]

    t_0 = np.array(f["t_0"][:])
    # (samples, features, r, lm) -> (samples, r, lm, features)
    f_m = np.moveaxis(np.array(f["f_m"][:]), 1, -1)

sub_trajs = int(len(f_m) / steps)
f_m = np.array(np.split(f_m, sub_trajs, axis=0))

seed = 42
key = jax.random.key(seed)


class CalibrationModel(nnx.Module):
    def __init__(self):
        # Initial guesses
        init_Ra, init_E, init_Pr = jax.random.uniform(
            key,
            shape=(3,),
            minval=jnp.array([10, 1e-2, 0.1]),
            maxval=jnp.array([50, 1e-1, 3]),
        )

        print(init_Ra, init_E, init_Pr)
        self.Ra = nnx.Param(jnp.log(init_Ra))
        self.E = nnx.Param(jnp.log(init_E))
        self.Pr = nnx.Param(jnp.log(init_Pr))

    def __call__(self, _x):
        return (jnp.exp(self.Ra), jnp.exp(self.E), jnp.exp(self.Pr))


model = CalibrationModel()

optimizer = nnx.Optimizer(model, optax.adam(2e-2), wrt=nnx.Param)


def kinetic_energy_loss(
    f_m_hat: jnp.ndarray,
    f_m: jnp.ndarray,
) -> np.ndarray:
    (
        _,
        u_tor_lm_hat,
        u_pol_lm_hat,
    ) = jnp.moveaxis(f_m_hat, -1, 0)
    (
        _,
        u_tor_lm,
        u_pol_lm,
    ) = jnp.moveaxis(f_m, -1, 0)

    l2 = eq.l_map * (eq.l_map + 1)

    u_tor_mis_lm = u_tor_lm_hat - u_tor_lm
    u_pol_mis_lm = u_pol_lm_hat - u_pol_lm

    eke_lm = kinetic_energy(eq, u_tor_mis_lm, u_pol_mis_lm)
    return jnp.sum(eke_lm[:, :, eq.sh.m != 0]) + jnp.sum(
        0.5 * eke_lm[:, :, eq.sh.m == 0]
    )


def flow_loss(
    model: nnx.Module, t_0: jnp.ndarray, f_0: jnp.ndarray, f_m: jnp.ndarray
) -> float:
    eq.Ra, eq.E, eq.Pr = model(None)

    def source(theta_lm, u_tor_lm, u_pol_lm):
        return (0, 0, 0)

    step_fn = jax.jit(
        dynamical_solver(
            eq,
            imex.PC2(dt),
            source,
        )
    )

    def __loop__(states: jnp.ndarray, cur_step: int) -> jnp.ndarray:
        theta_lm, u_tor_lm, u_pol_lm = states
        c, *states = step_fn(theta_lm, u_tor_lm, u_pol_lm)
        return (jnp.array(states), jnp.stack(states, axis=-1))

    _, f_m_hat = jax.lax.scan(__loop__, jnp.moveaxis(f_0, -1, 0), length=steps - 1)
    return kinetic_energy_loss(
        f_m_hat,
        f_m,
    )


@nnx.jit
def train_step(model, optimizer, traj_batch: jnp.ndarray):
    t_0, f_0, f_m = traj_batch
    loss, grads = nnx.value_and_grad(flow_loss)(model, t_0, f_0, f_m)
    optimizer.update(model, grads)
    return loss


# Batch generator
def batch_gen(t_0: np.ndarray, f_m: np.ndarray, sub_trajs: int, shuffle=True, key=None):
    """Generate trajectories (batch of continous samples) from a given dataset."""
    idx = np.arange(sub_trajs)
    if shuffle:
        idx = jax.random.permutation(key, idx, independent=True)

    for batch in range(sub_trajs):
        curr_idx = idx[batch]
        batch_inputs = f_m[curr_idx, 0]
        batch_target = f_m[curr_idx, 1:]
        yield (t_0[curr_idx], jnp.array(batch_inputs), jnp.array(batch_target))


Ra_train = []
E_train = []
Pr_train = []

train_loss = []
print("Calibrating model parameters...")
pbar = tqdm.tqdm(range(35), bar_format="{l_bar}{bar:10}{r_bar}{bar:-10b}")
for i in pbar:
    key, subkey = jax.random.split(key)
    data_sample = batch_gen(t_0, f_m, sub_trajs, shuffle=True, key=subkey)

    t_loss = 0.0
    for traj in range(sub_trajs):
        traj_batch = next(data_sample)
        loss = train_step(model, optimizer, traj_batch)
        t_loss += loss / (steps - 1)

        Ra_train.append(np.exp(model.Ra.get_value()))
        E_train.append(np.exp(model.E.get_value()))
        Pr_train.append(np.exp(model.Pr.get_value()))

        pbar.set_postfix(
            sub_traj=traj,
            loss=t_loss / (traj + 1),
            Ra=Ra_train[-1],
            E=E_train[-1],
            Pr=Pr_train[-1],
        )
    train_loss.append(t_loss / sub_trajs)

In [ ]:
fig, axs = plt.subplots(figsize=(12, 3), ncols=4)

axs[0].plot(train_loss)
axs[0].set_ylabel("Train loss")
axs[0].set_xlabel("Epoch")
axs[1].plot(Ra_train)
axs[1].axhline(100, color="tab:orange", linestyle="--")
axs[1].set_ylabel("Ra")
axs[1].set_xlabel("Iteration")
axs[2].semilogy(E_train)
axs[2].axhline(1e-3, color="tab:orange", linestyle="--")
axs[2].set_ylabel("E")
axs[2].set_xlabel("Iteration")
axs[3].semilogy(Pr_train)
axs[3].axhline(1, color="tab:orange", linestyle="--")
axs[3].set_ylabel("Pr")
axs[3].set_xlabel("Iteration")

fig.tight_layout()
plt.show()